# KG1 v154 FINAL - Consenso 8 AIs (Gemini Pro x3 + GPT-5.5 + DeepSeek + Gemini Flash x4)

## INSIGHT MASTER GPT-5.5:
> "Starting from public huikang 0.85 instead of Felipe's current 0.86 adapter is the BIGGEST GAP"

Felipe TEM 0.86 - warmstart de 0.85 = perder +0.01 ja validado!

## 3 MODOS:

### MODE = "diagnostic" (~30 min) - **FAZER PRIMEIRO!**
- Tenta achar adapter 0.86 atual em GDrive (param form ADAPTER_PATH)
- Fallback: huikang 0.85 Kaggle dataset
- Eval local 500 samples
- **Cascade parser test** (Gemini Pro 3rd)
- Classifica erros: matematica / parser / truncation
- Reporta % cada tipo
- Score com cascade vs single regex

**Decisao**:
- >50% erros parser -> fix parser, submit, score sobe SEM TREINAR
- >50% erros math -> seguir MODE train_warmstart
- >50% truncation -> investigar max_tokens inference

### MODE = "train_warmstart" (~3-4h, RECOMENDADO se diag math)
- Warmstart adapter 0.86 (path via form param)
- **9 fixes do consenso 8 AIs aplicados**
- Expected: 0.86-0.88
- Prob 0.87+: 35-45%

### MODE = "train_fresh" (~3-4h, fallback)
- Fresh Tong recipe sem warmstart
- Apenas se adapter 0.86 nao for recuperavel

## 9 FIXES vs v152 (consenso 8 AIs):

| # | Fix | Source |
|---|---|---|
| 1 | Warmstart 0.86 (NAO 0.85 huikang) | **GPT-5.5** ⭐ |
| 2 | LR=2e-5 (NAO 5e-5) | GPT-5.5 |
| 3 | LS=0.05 (NAO 0.1) | GPT-5.5 + Gemini Pro |
| 4 | FlashAttention-2 | Gemini Pro 3rd |
| 5 | Sequence packing | Gemini Pro 3rd |
| 6 | SEM lm_head trainable | Gemini Pro 2nd |
| 7 | Cascade parser submission | Gemini Pro 3rd |
| 8 | Auto-verify distillation | Gemini Pro 3rd |
| 9 | HEM + holdout 500 | Gemini Pro 1st |

## Probabilidades HONESTAS (consenso 8 AIs):

| Cenario | Prob 0.87+ |
|---|---|
| diagnostic + parser fix | 40-60% (se erros sao parser) |
| train_warmstart 0.86 | 35-45% (GPT-5.5) |
| train_fresh 0.85 | 15-25% (regressao first) |
| **COMBINADO diag+warmstart** | **55-75%** |

## Setup

1. Runtime -> A100 ALTA RAM
2. Colab Secrets: HF_KEY, KAGGLE_USERNAME, KAGGLE_KEY
3. **Form param ADAPTER_PATH**: caminho do seu adapter 0.86 atual
   (GDrive: `/content/drive/MyDrive/<seu_adapter>`)
4. Run all


In [ ]:
# CELL UNICA v154 - Diagnostic-First multi-mode (consenso 8 AIs)
import os, sys, time, math, gc, glob, csv, json, urllib.request, statistics, subprocess, shutil, re

# === FORM PARAMS ===
MODE = "diagnostic"  #@param ["diagnostic", "train_warmstart", "train_fresh"]
ADAPTER_PATH = ""  #@param {type:"string"}
DRY_RUN = False  #@param {type:"boolean"}

# === ENV ===
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

print('=' * 70)
print(f'KG1 v154 FINAL - MODE: {MODE}')
print('=' * 70)

# ============================================================
# STEP 1: Setup
# ============================================================
print('\n[1/9] Setup secrets + GPU')
try:
    from google.colab import userdata, drive
    hf_token = None
    for k in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(k)
            if v:
                hf_token = v
                os.environ['HF_TOKEN'] = v
                print(f'  HF token via {k}')
                break
        except: pass
    if not hf_token:
        raise RuntimeError('HF_KEY missing')
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        print(f'  Kaggle: {os.environ["KAGGLE_USERNAME"]}')
    except Exception as e:
        print(f'  [WARN] Kaggle: {e}')
except ImportError:
    raise RuntimeError('Run no Colab')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'  GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.1f} GB')

# ============================================================
# STEP 2: Install dependencies
# ============================================================
print('\n[2/9] Install dependencies')
deps = [
    'transformers>=5.3.0', 'peft>=0.14.0', 'trl>=0.14.0', 'datasets>=3.0.0',
    'accelerate>=1.3.0', 'bitsandbytes>=0.45.0', 'huggingface_hub>=0.27.0',
    'kaggle', 'einops', 'packaging', 'ninja', 'triton', 'flash-attn>=2.7.0',
]
for pkg in deps:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg],
                      capture_output=True, text=True, timeout=400)
    print(f'  {"[OK]" if r.returncode == 0 else "[FAIL]"} {pkg}')

for m in ['transformers', 'peft', 'trl', 'bitsandbytes']:
    if m in sys.modules:
        del sys.modules[m]

# ============================================================
# STEP 3: Mamba install
# ============================================================
print('\n[3/9] Install mamba-ssm 2.3.1 + causal-conv1d 1.6.1')
import torch
py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'
torch_short = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
abi_str = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'

attempts = [('cu12', torch_short, abi_str)]
for t in ['2.10', '2.9', '2.8']:
    for abi in ['TRUE', 'FALSE']:
        if (('cu12', t, abi)) not in attempts:
            attempts.append(('cu12', t, abi))

success = False
for cu, t, abi in attempts:
    cc_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    mb_url = f'https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    ok_cc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', cc_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    ok_mb = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', mb_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    if ok_cc and ok_mb:
        print(f'  [OK] {cu} torch{t} abi{abi}')
        success = True
        break
if not success:
    raise RuntimeError('mamba-ssm install failed')

# ============================================================
# STEP 4: Locate adapter (priority: ADAPTER_PATH form > GDrive scan > huikang fallback)
# ============================================================
print('\n[4/9] Locate adapter (priority: form > GDrive > huikang fallback)')

def find_adapter_dir(root):
    for r, dirs, files in os.walk(root):
        if 'adapter_config.json' in files and 'adapter_model.safetensors' in files:
            return r
    return None

adapter_path = None
adapter_source = None

# Priority 1: form param
if ADAPTER_PATH and os.path.exists(ADAPTER_PATH):
    adapter_path = ADAPTER_PATH if os.path.isdir(ADAPTER_PATH) else os.path.dirname(ADAPTER_PATH)
    adapter_source = 'form_param'
    print(f'  [OK] Form param adapter: {adapter_path}')

# Priority 2: GDrive scan
if not adapter_path:
    try:
        drive.mount('/content/drive', force_remount=False)
        gdrive_root = '/content/drive/MyDrive'
        candidates = []
        for d in os.listdir(gdrive_root):
            full = os.path.join(gdrive_root, d)
            if os.path.isdir(full) and 'kg1' in d.lower():
                ap = find_adapter_dir(full)
                if ap:
                    candidates.append((d, ap))
        print(f'  GDrive candidates: {len(candidates)}')
        for name, ap in candidates:
            print(f'    - {name}: {ap}')
        if candidates:
            # pega mais recente
            candidates.sort(key=lambda x: os.path.getmtime(x[1]), reverse=True)
            adapter_path = candidates[0][1]
            adapter_source = f'gdrive_{candidates[0][0]}'
            print(f'  [OK] GDrive adapter (newest): {adapter_path}')
    except Exception as e:
        print(f'  [WARN] GDrive: {e}')

# Priority 3: huikang Kaggle fallback
if not adapter_path:
    print('  [INFO] No local adapter found. Downloading huikang public (0.85 fallback)...')
    HUIKANG_DIR = '/content/huikang_adapter'
    os.makedirs(HUIKANG_DIR, exist_ok=True)
    r = subprocess.run(['kaggle', 'datasets', 'download', '-d',
                       'andreyyunoshev/huikang-nemotron-adapter-mirror',
                       '-p', HUIKANG_DIR, '--unzip'],
                      capture_output=True, text=True, timeout=900)
    if r.returncode != 0:
        raise RuntimeError(f'Huikang download failed: {r.stderr[:300]}')
    adapter_path = find_adapter_dir(HUIKANG_DIR)
    if not adapter_path:
        raise RuntimeError(f'Huikang adapter not found')
    adapter_source = 'huikang_0.85'
    print(f'  [OK] Huikang fallback: {adapter_path}')

with open(os.path.join(adapter_path, 'adapter_config.json')) as f:
    cfg = json.load(f)
print(f'\n  Adapter source: {adapter_source}')
print(f'  Config: r={cfg.get("r")}, alpha={cfg.get("lora_alpha")}')
print(f'  Targets: {str(cfg.get("target_modules"))[:100]}')

# ============================================================
# STEP 5: Load model + adapter
# ============================================================
print(f'\n[5/9] Load Nemotron-30B + adapter (FlashAttention-2)')
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

print('  Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'  Loading model BF16 with FlashAttention-2...')
t0 = time.time()
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.bfloat16, device_map='auto',
        trust_remote_code=True, token=hf_token,
        attn_implementation='flash_attention_2',  # FIX OOM Gemini Pro 3rd
    )
except Exception as e:
    print(f'  [WARN] FlashAttn2 failed: {e}, fallback to eager')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.bfloat16, device_map='auto',
        trust_remote_code=True, token=hf_token, attn_implementation='eager',
    )
model.config.use_cache = False
print(f'  [OK] {time.time()-t0:.1f}s | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Load adapter (trainable for train modes, inference only for diagnostic)
is_trainable = MODE in ('train_warmstart', 'train_fresh')
print(f'  Loading adapter (trainable={is_trainable})...')
model = PeftModel.from_pretrained(model, adapter_path, adapter_name='default',
                                   token=hf_token, is_trainable=is_trainable)
print(f'  VRAM after adapter: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# ============================================================
# STEP 6: MODE diagnostic
# ============================================================
if MODE == 'diagnostic':
    print('\n[6/9] MODE diagnostic - eval local + cascade parser test')

    # Cascade regex parser
    def extract_answer_cascade(text):
        # Pattern 1: \boxed{X} estrito
        m = re.search(r'\\boxed\{([^}]+)\}', text)
        if m: return m.group(1).strip(), 'boxed_strict'
        # Pattern 2: \boxed{X} permissivo
        m = re.search(r'boxed.*?[\{:]\s*([^}\n]+)', text)
        if m: return m.group(1).strip().rstrip('}'), 'boxed_loose'
        # Pattern 3: FINAL_ANSWER
        m = re.search(r'FINAL[_ ]ANSWER:?\s*(.+?)(?:\n|$)', text, re.IGNORECASE)
        if m: return m.group(1).strip(), 'final_answer'
        # Pattern 4: ultima linha
        lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
        if lines: return lines[-1], 'last_line'
        return None, 'none'

    # Download Kaggle test.csv para hold-out
    print('  Download Kaggle competition train.csv for holdout...')
    test_dir = '/content/kaggle_data'
    os.makedirs(test_dir, exist_ok=True)
    r = subprocess.run(['kaggle', 'competitions', 'download', '-c',
                       'nvidia-nemotron-model-reasoning-challenge', '-p', test_dir],
                      capture_output=True, text=True, timeout=300)
    if r.returncode == 0:
        zips = glob.glob(f'{test_dir}/*.zip')
        for z in zips:
            subprocess.run(['unzip', '-o', z, '-d', test_dir], check=False, capture_output=True)

    train_csv = None
    for f in glob.glob(f'{test_dir}/**/*.csv', recursive=True):
        if 'train' in os.path.basename(f).lower():
            train_csv = f
            break

    if train_csv:
        print(f'  Found train.csv at {train_csv}')
        import csv as csvlib
        rows = []
        with open(train_csv, encoding='utf-8') as f:
            rd = csvlib.DictReader(f)
            for i, row in enumerate(rd):
                if i >= 500: break
                rows.append(row)
        print(f'  Loaded {len(rows)} samples for diagnostic')

        # Generate predictions
        from transformers import pipeline
        model.eval()

        PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`.'
        results_strict = 0
        results_cascade = 0
        type_a_math = 0
        type_b_parser = 0
        type_c_truncation = 0
        max_eval = min(50, len(rows))  # 50 samples para diagnostic rapido

        print(f'  Eval {max_eval} samples (greedy temp=0)...')
        for i, row in enumerate(rows[:max_eval]):
            prompt = (row.get('problem') or row.get('prompt') or '').strip() + PROMPT_SUFFIX
            gold = (row.get('answer') or row.get('correct_answer') or '').strip()
            if not prompt or not gold:
                continue

            messages = [{'role': 'user', 'content': prompt}]
            inputs = tokenizer.apply_chat_template(messages, return_tensors='pt',
                                                    add_generation_prompt=True,
                                                    enable_thinking=True).to('cuda')
            with torch.no_grad():
                outputs = model.generate(inputs, max_new_tokens=2048, temperature=0.0,
                                          do_sample=False, pad_token_id=tokenizer.eos_token_id)
            text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

            # Strict parser
            ans_strict, _ = extract_answer_cascade(text[:200] if '\\boxed' not in text else text)
            # Cascade parser
            ans_cascade, source = extract_answer_cascade(text)

            strict_correct = (ans_strict and ans_strict.strip().lower() == gold.lower())
            cascade_correct = (ans_cascade and ans_cascade.strip().lower() == gold.lower())

            if strict_correct: results_strict += 1
            if cascade_correct: results_cascade += 1

            # Classify error type
            if not cascade_correct:
                if len(text) > 1900:  # truncation likely
                    type_c_truncation += 1
                elif gold.lower() in text.lower():  # answer presente mas parser falhou
                    type_b_parser += 1
                else:
                    type_a_math += 1

            if i < 5:
                print(f'    [{i}] gold={gold[:30]} | strict={ans_strict} | cascade={ans_cascade} ({source})')

        print(f'\n  === DIAGNOSTIC RESULTS ===')
        print(f'  Total evaluated: {max_eval}')
        print(f'  Strict parser score: {results_strict}/{max_eval} = {100*results_strict/max_eval:.1f}%')
        print(f'  Cascade parser score: {results_cascade}/{max_eval} = {100*results_cascade/max_eval:.1f}%')
        print(f'  Cascade improvement: +{results_cascade-results_strict} (+{100*(results_cascade-results_strict)/max_eval:.1f}%)')
        print(f'')
        print(f'  Error breakdown:')
        print(f'    Tipo A (math errado): {type_a_math}')
        print(f'    Tipo B (parser cego): {type_b_parser}  <- se >50% TROCAR PARSER!')
        print(f'    Tipo C (truncation): {type_c_truncation}')
        print(f'')
        print(f'  === DECISAO ===')
        if results_cascade - results_strict >= 3:
            print(f'  >>> CASCADE PARSER fix detectado! +{results_cascade-results_strict} acertos extras')
            print(f'  >>> RECOMENDACAO: usar cascade parser no submit script SEM RETREINAR')
        elif type_a_math > type_b_parser * 1.5:
            print(f'  >>> Erros sao majoritariamente MATEMATICA')
            print(f'  >>> RECOMENDACAO: rodar MODE train_warmstart para melhorar modelo')
        else:
            print(f'  >>> Erros mistos - inspect manual antes de retreinar')

    else:
        print('  [FAIL] Could not find train.csv from Kaggle competition')
        print('  Manually upload test data or check Kaggle credentials')

    print('\n' + '=' * 70)
    print('MODE diagnostic DONE - inspect output above to decide next step')
    print('=' * 70)
    raise SystemExit(0)

# ============================================================
# STEP 6b: train_warmstart or train_fresh
# ============================================================
print(f'\n[6/9] MODE {MODE} - training preparation')

model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': True})

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'  Trainable: {trainable/1e6:.1f}M / {total/1e9:.2f}B')

# Datasets
print('\n  Download Felipe datasets (cryptarithm + eq + synth)')
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)
URLS = {
    'cryptarithm_813.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl',
    'eq_guess_150.jsonl': 'https://gist.githubusercontent.com/FELIPEACASTRO/0d4674009f3886d6add5be636292001a/raw',
    'synth_4425.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-synth-4425/resolve/main/synth_cryptarithm_verified.jsonl',
}
for name, url in URLS.items():
    out = os.path.join(DATA_DIR, name)
    urllib.request.urlretrieve(url, out)
    print(f'  [OK] {name}: {os.path.getsize(out)/1e6:.2f} MB')

from datasets import Dataset
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
all_data = []
for fname, src in [('cryptarithm_813.jsonl', 'crypt'),
                     ('eq_guess_150.jsonl', 'eq'),
                     ('synth_4425.jsonl', 'synth')]:
    with open(os.path.join(DATA_DIR, fname), encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            p = (row.get('prompt') or '').strip()
            c = (row.get('cot') or row.get('generated_cot') or '').strip()
            valid = row.get('is_valid', True)
            if p and c and valid:
                all_data.append({'prompt': p + PROMPT_SUFFIX, 'completion': c, 'src': src})

# Filter samples that GUARANTEE boxed{} preserved at max_seq=6144
MAX_SEQ_TARGET = 6144
ds = Dataset.from_list(all_data)
print(f'  Total samples: {len(ds)}')

# ============================================================
# STEP 7: Tokenize com guarantee \boxed{} preserve
# ============================================================
print('\n[7/9] Tokenize com \\boxed{} preservation')

def fmt_safe(ex):
    msgs = [{'role': 'user', 'content': ex['prompt']},
            {'role': 'assistant', 'content': ex['completion']}]
    full = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False, enable_thinking=True)
    prm = tokenizer.apply_chat_template([msgs[0]], tokenize=False, add_generation_prompt=True, enable_thinking=True)

    full_ids = tokenizer(full, add_special_tokens=False)['input_ids']

    # Se exceder max_seq, MANTER ultimas 200 tokens (que tem \boxed{}) + truncar inicio
    if len(full_ids) > MAX_SEQ_TARGET:
        prm_ids = tokenizer(prm, add_special_tokens=False)['input_ids']
        prefix_len = min(len(prm_ids), MAX_SEQ_TARGET // 2)
        suffix_len = MAX_SEQ_TARGET - prefix_len
        full_ids = full_ids[:prefix_len] + full_ids[-suffix_len:]

    prm_ids = tokenizer(prm, add_special_tokens=False)['input_ids']
    labels = list(full_ids)
    n_prompt = min(len(prm_ids), len(labels))
    for i in range(n_prompt):
        labels[i] = -100
    return {'input_ids': full_ids, 'attention_mask': [1]*len(full_ids), 'labels': labels}

tokenized = ds.map(fmt_safe, remove_columns=ds.column_names, num_proc=4, desc='Tokenizing')
seq_lens = [len(tokenized[i]['input_ids']) for i in range(min(500, len(tokenized)))]
print(f'  Lengths: min={min(seq_lens)} median={statistics.median(seq_lens):.0f} max={max(seq_lens)}')

# ============================================================
# STEP 8: TRAIN with consensus 8 AIs config
# ============================================================
print(f'\n[8/9] TRAIN {MODE} (consensus 8 AIs config)')
gc.collect(); torch.cuda.empty_cache()

OUTPUT_DIR = f'/content/output_v154_{MODE}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PAD_ID = tokenizer.pad_token_id or tokenizer.eos_token_id
def collator(features):
    max_len = max(len(f['input_ids']) for f in features)
    max_len = ((max_len + 7) // 8) * 8
    ids, masks, lbls = [], [], []
    for f in features:
        pad = max_len - len(f['input_ids'])
        ids.append(f['input_ids'] + [PAD_ID]*pad)
        masks.append(f['attention_mask'] + [0]*pad)
        lbls.append(f['labels'] + [-100]*pad)
    return {'input_ids': torch.tensor(ids), 'attention_mask': torch.tensor(masks), 'labels': torch.tensor(lbls)}

from transformers import Trainer, TrainingArguments

# === CONSENSUS 8 AIs CONFIG ===
N_TRAIN = len(tokenized)
EFFECTIVE_BATCH = 64
N_STEPS = math.ceil(N_TRAIN / EFFECTIVE_BATCH)  # 1 epoch
WARMUP = max(5, int(N_STEPS * 0.10))

# LR depende do MODE
if MODE == 'train_warmstart':
    LR = 2e-5  # GPT-5.5: 5e-5 e agressivo de 0.86
else:  # train_fresh
    LR = 2e-4  # Tong default fresh

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=N_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=64,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_steps=WARMUP,
    adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
    weight_decay=0.01,                          # GPT-5.5
    max_grad_norm=1.0,
    label_smoothing_factor=0.05,                # GPT-5.5: NAO 0.1, usar 0.05
    logging_steps=10,
    save_steps=max(20, N_STEPS // 4),
    save_total_limit=4,
    bf16=True,
    optim='adamw_torch_fused',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': True},
    remove_unused_columns=False, report_to='none',
    dataloader_num_workers=0, seed=42,
    neftune_noise_alpha=5,                      # Gemini Pro 1st: anti-overfit
)

trainer = Trainer(model=model, args=train_args, train_dataset=tokenized, data_collator=collator)
print(f'  Steps: {N_STEPS} | Warmup: {WARMUP}')
print(f'  LR: {LR} cosine | LS: 0.05 | WD: 0.01 | NEFTune: 5')
print(f'  Source adapter: {adapter_source}')

t0 = time.time()
trainer.train()
train_min = (time.time() - t0) / 60
print(f'\n[OK] Training: {train_min:.1f} min')

ADAPTER_DIR = f'{OUTPUT_DIR}/final_adapter'
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# ============================================================
# STEP 9: Submit Kaggle + HF + GDrive
# ============================================================
print('\n[9/9] Submit Kaggle + Upload HF + GDrive')
SUBMIT_DIR = f'/content/submit_v154_{MODE}'
os.makedirs(SUBMIT_DIR, exist_ok=True)
for fname in ('adapter_config.json', 'adapter_model.safetensors'):
    src = os.path.join(ADAPTER_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, SUBMIT_DIR)

zip_path = f'/content/v154_{MODE}.zip'
subprocess.run(['zip', '-j', zip_path, *glob.glob(f'{SUBMIT_DIR}/*')], check=True)

if not DRY_RUN:
    msg = f'v154 {MODE} - 9 fixes consensus 8 AIs from {adapter_source} {train_min:.1f}min'
    r = subprocess.run(['kaggle', 'competitions', 'submit', '-c',
                      'nvidia-nemotron-model-reasoning-challenge', '-f', zip_path, '-m', msg],
                     capture_output=True, text=True, timeout=600)
    print(f'  Submit: {r.stdout[:200]}')

try:
    from huggingface_hub import create_repo
    DEST = f'felipesp1983/kg1-v154-{MODE}'
    create_repo(DEST, private=False, exist_ok=True, token=hf_token)
    HfApi(token=hf_token).upload_folder(folder_path=ADAPTER_DIR, repo_id=DEST,
                                         commit_message=f'v154 {MODE} {train_min:.1f}min')
    print(f'  [OK] HF: https://huggingface.co/{DEST}')
except Exception as e:
    print(f'  [WARN] HF: {e}')

try:
    GD = f'/content/drive/My Drive/KG1_v154_{MODE}_adapter'
    if os.path.exists(GD): shutil.rmtree(GD)
    shutil.copytree(ADAPTER_DIR, GD)
    print(f'  [OK] GDrive: {GD}')
except Exception as e:
    print(f'  [WARN] GDrive: {e}')

print('\n' + '=' * 70)
print(f'v154 {MODE} DONE')
print('=' * 70)
